# Proyecto Final - Sistemas de Recomendacion
Autor: Adrián Robles Arques

En este proyecto, vamos a implementar un sistema de recomendación basado en filtrado colaborativo, empleando para ello la librería Surprise. Como ya vimos en el módulo anterior, esta librería se especializa en sistemas de recomendación colaborativos con valoraciones explícitas, por tanto emplearemos un dataset de prueba acorde.

## Fase 1: Implementación del problema propuesto con Surprise

En este apartado realizamos la implementación del problema propuesto con Surprise.
Para ello, nos basamos en el problema que se plantea en el artículo de Medium correspondiente:
https://layla-scheli.medium.com/iebs-proyecto-final-sistemas-de-recomendacion-2024-915d35a62385

In [11]:
# Importamos las librerías que vamos a utilizar
import pandas as pd
from surprise import KNNBasic, KNNWithMeans
from surprise import Dataset
from surprise import Reader
from surprise import accuracy
from surprise.model_selection import train_test_split

In [12]:
# Realizamos la ingesta de datos
reader = Reader(line_format='user item rating timestamp', sep='::')
data = Dataset.load_from_file('C:\\Users\\demad\\Desktop\\Test\\DataScienceIEBS\\Bloque 7\\Sistema de Recomendacion\\datos\\ratings.dat', reader=reader)

In [13]:
# Repartimos los datos en dos subconjuntos, uno para entrenamiento y otro para test
train_data, test_data = train_test_split(data, test_size=0.3)

In [14]:
# Definimos los algoritmos de recomendación que vamos a utilizar
# KNNBasic utiliza la distancia de Pearson y KNNWithMeans utiliza la distancia coseno
knn = KNNBasic(k=50, sim_options={'name': 'pearson', 'user_based': True})
kMeans = KNNWithMeans(k=50, sim_options={'name': 'cosine','user_based': False})

# Entrenamos los modelos con los datos de entrenamiento
knn.fit(train_data)
kMeans.fit(train_data)

Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.


In [32]:
# Implementemos una prueba manual
user_id = 1
item_id = 6

# Probemos la predicción de ambos modelos
knn_user_prediction = knn.predict(user_id, item_id)
kMeans_user_prediction = kMeans.predict(user_id, item_id)

# Mostramos las predicciones
print(f"Predicción KNN Básico para el usuario {user_id} y el ítem {item_id}: {knn_user_prediction.est}")
print(f"Predicción KNN Promediado para el usuario {user_id} y el ítem {item_id}: {kMeans_user_prediction.est}")


Predicción KNN Básico para el usuario 1 y el ítem 6: 3.5803960888157613
Predicción KNN Promediado para el usuario 1 y el ítem 6: 3.5803960888157613


In [16]:
# Ahora evaluaremos los modelos con el conjunto de test
knn_test_predictions = knn.test(test_data)
kMeans_test_predictions = kMeans.test(test_data)

RMSE: 0.9613
RMSE: 0.8944


In [20]:
# Calculamos el RMSE para ambos modelos
knn_rmse = accuracy.rmse(knn_test_predictions)
kMeans_rmse = accuracy.rmse(kMeans_test_predictions)

# Mostramos los resultados de las evaluaciones
print(f"KNN RMSE:  {round(knn_rmse, 5)}")
print(f"KMeans RMSE: {round(kMeans_rmse, 5)}")

RMSE: 0.9613
RMSE: 0.8944
KNN RMSE:  0.96135
KMeans RMSE: 0.89442


## Fase 2: Análisis de resultados

En la sección previa hemos procedido con la implementación de dos métodos de recomendación basados en el algoritmo de primeros vecinos (K-nearest neighbors), uno empleando el sistema básico y el otro empleando un sistema de ajuste al centroide (media de datos) de cada uno de los grupos de clasificación. 

En la fase de testeo, el sistema simple ha obtenido un valor de RMSE superior al otro, lo que en principio indicaría que el algorimo de KMeans es más adecuado. Sin embargo, la diferencia en el error es pequeña, y el hecho de que la configuración empleada para el entrenamiento de ambos modelos sea diferente implica que no son directamente comparables, principalmente porque están empleando distintas métricas.

Una opción para evaluar cuál es el mejor modelo sería implementar un Grid Search, función que es compatible con la librería Surprise, y realizar un ajuste con diferentes hiperparámetros para cada modelo, buscando así el que mejor resultado obtenga para cada uno. Otra forma más simple es bien probar la métrica de pearson y la del coseno para ambos modelos, y ver cuál es el que mejor resultado obtiene.

## Fase 3: Artículo en Medium

Enlace al artículo: https://medium.com/@demadrian_3477/recomendaci%C3%B3n-de-pel%C3%ADculas-con-la-librer%C3%ADa-surprise-73bcd556f82b

## Fase opcional: Actualización del sistema de recomendación previo usando Surprise.

In [21]:
# importamos otras librerías necesarias
import numpy as np

In [22]:
# Cargamos los datasets de películas y usuarios
# Cargamos el dataset
peliculas = pd.read_csv('datos\\movies.csv')
peliculas.columns = ['PeliculaID', 'titulo', 'generos']

ratings = pd.read_csv('datos\\ratings.csv')
ratings.columns = ['UsuarioID', 'PeliculaID', 'Valoracion', 'Tiempo']

# Vamos a extraer todos los ID de los usuarios únicos
usuarios_tot = ratings['UsuarioID'].unique()

In [48]:
# Incorporamos funciones implementadas para generar las recomendaciones

# Vamos a definir la función para calcular la distancia entre dos usuarios cualesquiera tal como lo hemos definido

def dist_usuarios(user1, user2, minimo_peliculas=5, ratings=ratings):
    """Calcula la distancia euclidiana entre dos usuarios basándose en las valoraciones de películas que ambos han visto.
    
    Args:
        user1: ID del primer usuario. Int
        user2: ID del segundo usuario. Int
        minimo_peliculas: Número mínimo de películas que ambos usuarios deben haber visto para calcular la distancia. Int
        ratings: DataFrame de valoraciones de películas. DataFrame[UsuarioID, PeliculaID, Valoracion, Tiempo]
    
    Returns:
        La Distancia euclidiana entre los dos usuarios o None si no hay suficientes películas en común.
    """
    valoracion_u1 = ratings.query('UsuarioID==%d' % user1)[['PeliculaID', 'Valoracion']]
    valoracion_u2 = ratings.query('UsuarioID==%d' % user2)[['PeliculaID', 'Valoracion']]
    
    # Creamos un dataframe con las dos series de notas para las películas que hayan visto ambos
    valoracion_u1_u2 = pd.merge(valoracion_u1, valoracion_u2, on='PeliculaID', how='inner', suffixes=('_u1', '_u2'))
    
    if len(valoracion_u1_u2) < minimo_peliculas:
        return None
    
    # Extraemos los vectores
    x = np.array(valoracion_u1_u2['Valoracion_u1'])
    y = np.array(valoracion_u1_u2['Valoracion_u2'])
    
    # Calculamos la distancia entre los dos usuarios
    return np.linalg.norm(x - y)

def k_mas_similares(user_obj, usuarios, us_similares=1, minimo_peliculas=5, ratings=ratings):
    """
    Retorna el usuario mas similar al usuario ingresado
    
    Args:
        user_obj: Objeto de usuario
        usuarios: Lista de usuarios
        us_similares: Número de usuarios similares a retornar, por defecto 1
        Para dist_usuarios:
            minimo_peliculas: Número mínimo de películas compartidas. Int
            ratings: DataFrame de valoraciones
    Returns:
        Usuarios mas similarres al usuario objetivo y la distancia. Dataframe
    """
    # Excluimos el usuario objetivo de la lista de usuarios
    usuarios_ex = list(usuarios.copy())
    usuarios_ex.remove(user_obj)
    
    distancias = {}
    for usuario in usuarios_ex:
        dist = dist_usuarios(user_obj, usuario, minimo_peliculas=minimo_peliculas, ratings=ratings)
        distancias[usuario] = dist
    
    # Eliminamos aquellos que tengan valor None
    distancias = {k: v for k, v in distancias.items() if v is not None}
    
    # Ordenamos la lista de usuarios por su distancia
    usuarios_ordenados = np.array(sorted(distancias.items(), key=lambda x: x[1])).reshape(-1, 2)
    result = pd.DataFrame(usuarios_ordenados[:us_similares], columns=['UsuarioID', 'Distancia_user_ref'])
    result['UsuarioID'] = result['UsuarioID'].astype('Int64')
    result.set_index('UsuarioID', inplace=True)
    
    return result

def peliculas_no_vistas_por_usuario(user_obj, usuarios_similares):
    """
    Retorna las películas que han visto los usuarios similares al usuario objetivo y que este no ha visto.
    
    Args:
        user_obj: ID del usuario objetivo.
        usuarios_similares: DataFrame con los usuarios similares y sus distancias.
        
    Returns:
        DataFrame con las películas no vistas por el usuario objetivo.
    """
    # Extraemos las películas vistas por el usuario objetivo
    peliculas_vistas = ratings.query('UsuarioID==%d' % user_obj)['PeliculaID'].unique()
    
    # Creamos un DataFrame para almacenar las películas no vistas
    peliculas_no_vistas = pd.DataFrame()
    
    for usuario in usuarios_similares.index:
        # Extraemos las películas vistas por el usuario similar
        peliculas_usuario = ratings.query('UsuarioID==%d' % usuario)[['PeliculaID', 'Valoracion']]
        
        # Filtramos las películas que el usuario objetivo no ha visto
        peliculas_usuario_no_vistas = peliculas_usuario[~peliculas_usuario['PeliculaID'].isin(peliculas_vistas)]
        
        # Añadimos al DataFrame de películas no vistas
        peliculas_no_vistas = pd.concat([peliculas_no_vistas, peliculas_usuario_no_vistas])
    
    return peliculas_no_vistas.reset_index(drop=True)

# Vamos a recomendar las películas no vistar por el usuario objetivo
# Vamos a probar a añadir filtrado por género
def recomendar_peliculas(user_obj, usuarios, peliculas, usuarios_similares = 5, num_recomendaciones=5,
                        minimo_peliculas=5, ratings=ratings):
    """
    Recomienda películas al usuario objetivo basándose en las valoraciones de usuarios similares.
    
    Args:
        user_obj: ID del usuario objetivo. Int
        usuarios: Lista de IDs de usuarios. Array-like(Int)
        peliculas: DataFrame con las películas y sus géneros. DataFrame
        usuarios_similares: Cantidad de usuarios similares a considerar. Int
        num_recomendaciones: Número de recomendaciones a retornar.
        minimo_peliculas: Número mínimo de películas que deben haber visto los usuarios similares. Int
        ratings: DataFrame de valoraciones de películas. DataFrame[UsuarioID, PeliculaID, Valoracion, Tiempo]
    
    Returns:
        DataFrame con las películas recomendadas y sus valoraciones promedio.
    """
    usr_sim = k_mas_similares(user_obj, usuarios, usuarios_similares,
                                minimo_peliculas=minimo_peliculas, ratings=ratings)
    
    peliculas_no_vistas = peliculas_no_vistas_por_usuario(user_obj, usr_sim)
    
    # Agrupamos por PeliculaID y calculamos la valoración promedio
    recomendaciones = peliculas_no_vistas.groupby('PeliculaID')['Valoracion'].mean().reset_index()
    
    # Ordenamos por valoración promedio y seleccionamos las mejores
    recomendaciones.sort_values(by='Valoracion', ascending=False, inplace=True)
    recomendaciones.set_index('PeliculaID', inplace=True)
    
    
    peliculas_recomendadas = pd.merge(recomendaciones,
                                    peliculas.reset_index()[['PeliculaID','titulo', 'generos']],
                                    left_index=True, right_on='PeliculaID', how='left')
    
    # Devolvemos la cantidad fijada de recomendaciones
    peliculas_recomendadas = peliculas_recomendadas.head(num_recomendaciones)
    return peliculas_recomendadas[['titulo', 'generos', 'Valoracion']].reset_index(drop=True)

In [64]:
# Hagamos una prueba de uso
recomendar_peliculas(421, usuarios_tot, peliculas, num_recomendaciones=10)

,titulo,generos,Valoracion
0,300 (2007),Action|Fantasy|War|IMAX,5.0
1,Dead Man (1995),Drama|Mystery|Western,5.0
2,Black Swan (2010),Drama|Thriller,5.0
3,Stand by Me (1986),Adventure|Drama,5.0
4,True Grit (2010),Western,5.0
5,Monty Python's The Meaning of Life (1983),Comedy,5.0
6,Louis C.K.: Hilarious (2010),Comedy,5.0
7,Fargo (1996),Comedy|Crime|Drama|Thriller,5.0
8,Louis C.K.: Chewed Up (2008),Comedy,5.0
9,Louis C.K.: Shameless (2007),Comedy,5.0


¿Cómo podemos mejorar el actual sistema de recomendación?

* Podemos icorporar el uso de KNN Means para calcular la valoración estimada. (Surprise)
* Podemos emplear la columna de género para mejorar el sistema de recomendación, usando un filtrado por género.
* Podemos añadir la columna de año de publicación para mejorar el sistema de recomendación.

In [ ]:
# Preparamos el Dataset para el modelo de Surprise
dataset = Dataset.load_from_df(ratings[['UsuarioID', 'PeliculaID', 'Valoracion']], Reader(rating_scale=(1, 5)))

# Repartimos los datos en dos subconjuntos, uno para entrenamiento y otro para test
train_data, test_data = train_test_split(dataset, test_size=0.3)

# Definimos el modelo KNN con medias, con métrica de Pearson y basado en usuarios
kMeans = KNNWithMeans(k=50, sim_options={'name': 'MSD', 'user_based': True, 'min_support': 5})

In [87]:
# Entrenamos el modelo con los datos de entrenamiento
kMeans.fit(train_data)

Computing the msd similarity matrix...
Done computing similarity matrix.


In [89]:
# Testeamos los resultados del modelo
kMeans_test_predictions = kMeans.test(test_data)
# Calculamos el RMSE para el modelo KNN con medias
kMeans_rmse = accuracy.rmse(kMeans_test_predictions)
# Mostramos el RMSE del modelo KNN con medias
print(f"KNN con medias RMSE: {round(kMeans_rmse, 5)}")

RMSE: 0.9043
KNN con medias RMSE: 0.90432


In [90]:
# Probamos la predicción de un usuario y una película
user_id = 10
item_id = 562
kMeans_user_prediction = kMeans.predict(user_id, item_id)
print(kMeans_user_prediction)

user: 10         item: 562        r_ui = None   est = 3.39   {'actual_k': 10, 'was_impossible': False}


In [99]:
# Actualizamos la función de recomendación para usar el modelo entrenado
def recomendar_peliculas_surprise(user_obj, peliculas, num_recomendaciones=5, Generos=None, anno= None):
    """_summary_
    Esta función recomienda películas a partir de los datos de entrenamiento del modelo KNNWithMeans de la librería
    Surprise. Implementa la posibilidad de filtrar las recomendaciones por géneros y por año de publicación.

    Args:
        user_obj: ID del usuario objetivo. Int
        peliculas: DataFrame con las películas y sus géneros. DataFrame
        num_recomendaciones: Número de recomendaciones a retornar.
        Generos (Array(Strings), Opcional): Lista de géneros a filtrar las recomendaciones. Default como None.
        anno (Int, Opcional): Año mínimo de publicación de las películas. Defaults como None.

    Returns:
        DataFrame con las películas recomendadas y sus valoraciones promedio.
    """
    # Extraemos la lista de películas no vistas en la lista de películas totales
    peliculas_vistas = ratings.query('UsuarioID==%d' % user_obj)['PeliculaID'].unique() # Peliculas vistas
    mask_vistas = peliculas['PeliculaID'].isin(peliculas_vistas) # Mascara de peliculas vistas
    peliculas_no_vistas = peliculas[mask_vistas == False].reset_index(drop=True) # Invertimos la máscara
        
    # Usamos el modelo KNNWithMeans para predecir las valoraciones de las películas no vistas
    prediccion = []
    for index, row in peliculas_no_vistas.iterrows():
        pred = kMeans.predict(user_obj, row['PeliculaID'])
        prediccion.append((row['PeliculaID'], pred.est))
        
    # Creamos un DataFrame con las películas no vistas y sus predicciones
    recomendaciones = pd.DataFrame(prediccion, columns=['PeliculaID', 'Valoracion'])
    recomendaciones['Valoracion'] = recomendaciones['Valoracion'].astype(float)
    
    # Ordenamos por valoración promedio y seleccionamos las mejores
    recomendaciones.sort_values(by='Valoracion', ascending=False, inplace=True)
    recomendaciones.set_index('PeliculaID', inplace=True)
    
    peliculas_recomendadas = pd.merge(recomendaciones,
                                    peliculas.reset_index()[['PeliculaID','titulo', 'generos']],
                                    left_index=True, right_on='PeliculaID', how='left')
    
    # Vamos a crear una columna para recoger el año de publicación de la película
    # Teniendo en cuenta que el año de publicación aparece en el título entre paréntesis
    peliculas_recomendadas['Año'] = peliculas_recomendadas['titulo'].str.extract(r'\((\d{4})\)')[0]
    peliculas_recomendadas['Año'] = pd.to_numeric(peliculas_recomendadas['Año'], errors='coerce').astype('Int64')
    
    # Borramos el año de publicación del título
    peliculas_recomendadas['titulo'] = peliculas_recomendadas['titulo'].str.replace(r'\s*\(\d{4}\)', '', regex=True)
    
    # Filtramos por géneros si se especifican
    if Generos:
        # El método 'apply' se usa para aplicar una función a cada fila del DataFrame que genera una máscara booleana
        # que indica si al menos uno de los géneros de la película está en la lista
        mask = peliculas_recomendadas.apply(lambda x: any(g in Generos for g in x['generos'].split('|')), axis=1)
        
        # La máscara es un array booleano que indica si cada fila cumple la condición
        # Aplicamos la máscara para filtrar las películas recomendadas
        peliculas_recomendadas = peliculas_recomendadas[mask]
        
    # Hacemos la comprobación por año de publicación
    if anno:
        # Filtramos las películas por el año de publicación
        peliculas_recomendadas = peliculas_recomendadas[peliculas_recomendadas['Año'] >= anno]
    
    # Devolvemos la cantidad fijada de recomendaciones
    peliculas_recomendadas = peliculas_recomendadas.head(num_recomendaciones)
    
    # Ajustamos el nombre de las columnas para la salida, ajustando el uso de mayúsculas y tildes
    peliculas_recomendadas = peliculas_recomendadas.rename(columns={'generos': 'Géneros', 'titulo': 'Título', 'Valoracion': 'Valoración'})
    
    return peliculas_recomendadas[['Título', 'Año', 'Géneros', 'Valoración']].reset_index(drop=True)

In [102]:
# Probamos con la solicitud anterior
recomendar_peliculas_surprise(421, peliculas, num_recomendaciones=10)

,Título,Año,Géneros,Valoración
0,Louis C.K.: Live at the Beacon Theater,2011,Comedy,5.0
1,The Bremen Town Musicians,1969,Animation|Drama|Fantasy,5.0
2,SORI: Voice from the Heart,2016,Drama|Sci-Fi,5.0
3,Hour of the Wolf (Vargtimmen),1968,Drama|Horror,5.0
4,"Doctor, The",1991,Drama,5.0
5,City of Men (Cidade dos Homens),2007,Drama,5.0
6,Pride,2014,Comedy|Drama,5.0
7,Son of Rambow,2007,Children|Comedy|Drama,5.0
8,3 Women (Three Women),1977,Drama,5.0
9,On the Beach,1959,Drama,5.0


In [101]:
# Probamos el filtrado por géneros y año de publicación
recomendar_peliculas_surprise(610, peliculas, num_recomendaciones=10, Generos=['Drama', 'Romance'], anno=2010)

,Título,Año,Géneros,Valoración
0,Blue Is the Warmest Color (La vie d'Adèle),2013,Drama|Romance,5.0
1,Honey (Miele),2013,Drama,5.0
2,"Separation, A (Jodaeiye Nader az Simin)",2011,Drama,5.0
3,Frank,2014,Comedy|Drama|Mystery,5.0
4,Chinese Puzzle (Casse-tête chinois),2013,Comedy|Romance,5.0
5,There Will Come a Day,2013,Comedy|Drama,5.0
6,All Yours,2016,Comedy|Drama|Romance,5.0
7,Oh Boy (A Coffee in Berlin),2012,Comedy|Drama,5.0
8,The Wait,2015,Drama,5.0
9,Faster,2010,Action|Crime|Drama,5.0


Como podemos ver la implementación del algoritmo de KNN means desde la librería Surprise es sencillo e intuitivo, a penas requiere un preprocesado mínimo de los datos de entrenamiento y el de prueba. Sin embargo ofrece una solución muy potente que acelera en gran medida la ejecución de la función de recomendación de películas. Además permite la implementación directa de métricas para evaluar la calidad de los resultados obtenidos. El principal inconveniente que encuentro es no poder alterar el número de vecinos que se consideran para la clasificación en la ejecución del algoritmo, dado que este es un parámetro de entrada del entrenamiento del modelo. Sin embargo esto no parece ser una desventaja relevante.

Una opción para mejorar este modelo sería aplicar un Search Grid al modelo de KNN promediado para refinar el uso de hiperparámetros que se consideran para el entrenamiento del modelo. Sin embargo estos deberían actualizarse cuando empleemos datos diferentes.

El hecho de que los resultados obtenidos difieran en gran medida por los devueltos por el algoritmo anterior podría explicarse por el cambio de métricas empleadas y de modelo, ya que el implementado manualmente es el KNN básico, sin calcular el centroide o promedio, y teniendo en cuenta que además la predicción ha cambiado en diferentes ejecuciones. Uno de los parámetros que se pueden considerar en un posible Grid Search sería la métrica a emplear y el número de vecinos a tener en cuenta.